In [1]:
import requests
import pandas as pd
# =========================
# Import paths and working directory
# import plugins from project
# =========================

import sys
sys.path.append(r"D:/aviation_data_pipeline")
import os
os.chdir(r"D:\aviation_data_pipeline")
# =========================
# Import env variables for DB connection into HOST OS
# load via dotenv()
# =========================
from dotenv import load_dotenv
load_dotenv()

True

## Cleaning Data
First, we set up the environment, establish the database connection, and query the required data.

In [2]:
from sqlalchemy import create_engine
from plugins.utils.connections import get_postgres_conn_str

In [3]:
pip show python-dotenv

Name: python-dotenvNote: you may need to restart the kernel to use updated packages.

Version: 0.21.0
Summary: Read key-value pairs from a .env file and set them as environment variables
Home-page: https://github.com/theskumar/python-dotenv
Author: Saurabh Kumar
Author-email: me+github@saurabh-kumar.com
License: BSD-3-Clause
Location: C:\Users\dimod\Anaconda3\Lib\site-packages
Requires: 
Required-by: anaconda-cloud-auth, pydantic-settings


In [4]:
engine = create_engine(get_postgres_conn_str())

In [5]:
staged_data = pd.read_sql(
    "SELECT * FROM data", con=engine
)
staged_data.head()

,time,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,1778835603,3ffc29,DMFSS,Germany,1778835600,1778835602,12.5225,48.4242,1059.18,False,45.79,270.64,-0.33,None,NaN,7000,False,0
1,1778835603,511187,MBU1RD,Estonia,1778835602,1778835602,12.2108,51.4197,NaN,True,5.14,104.06,NaN,None,NaN,1000,False,0
2,1778835603,511171,MBU4HW,Estonia,1778835603,1778835603,10.3395,50.4433,8808.72,False,237.14,34.03,-9.10,None,8549.64,1000,False,0
3,1778835603,511170,MBU1TF,Estonia,1778835603,1778835603,9.4142,49.8205,10980.42,False,245.48,10.63,0.00,None,10751.82,1000,False,0
4,1778835603,3d381b,DETMP,Germany,1778835602,1778835602,12.8749,48.8068,891.54,False,42.79,108.22,0.33,None,853.44,None,False,0


Now we can start to clean the data.<br>Since the data requested from the <u>OpenSky Network</u> API is already relatively clean at ingestion time, this step will remain intentionally lightweight. It can be extended or adjusted later if needed.<br>For now, we will drop unnecessary columns, strip whitespace from string-based fields, remove duplicates, and filter out rows missing key identifiers such as **icao24** or **callsign**

In [6]:
staged_data.shape

(591, 18)

In [12]:
drop_cols = ["sensors", "spi", "position_source"]
strip_cols =["icao24", "callsign", "origin_country"]

cleaned_data = staged_data.copy()

cleaned_data[strip_cols] = (
    cleaned_data[strip_cols]
    .astype(str)
    .apply(lambda col: col.str.strip())
)

cleaned_data = (
    cleaned_data
    .drop(columns=drop_cols)
    .drop_duplicates()
    .dropna(subset=["icao24", "callsign"])
)

In [13]:
staged_data

,time,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,1778835603,3ffc29,DMFSS,Germany,1778835600,1778835602,12.5225,48.4242,1059.18,False,45.79,270.64,-0.33,None,NaN,7000,False,0
1,1778835603,511187,MBU1RD,Estonia,1778835602,1778835602,12.2108,51.4197,NaN,True,5.14,104.06,NaN,None,NaN,1000,False,0
2,1778835603,511171,MBU4HW,Estonia,1778835603,1778835603,10.3395,50.4433,8808.72,False,237.14,34.03,-9.10,None,8549.64,1000,False,0
3,1778835603,511170,MBU1TF,Estonia,1778835603,1778835603,9.4142,49.8205,10980.42,False,245.48,10.63,0.00,None,10751.82,1000,False,0
4,1778835603,3d381b,DETMP,Germany,1778835602,1778835602,12.8749,48.8068,891.54,False,42.79,108.22,0.33,None,853.44,None,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586,1778835603,4b17e3,SWR7AK,Switzerland,1778835595,1778835603,8.5678,47.4550,525.78,True,0.00,129.38,NaN,None,NaN,2000,False,0
587,1778835603,4b17fc,SWR2WX,Switzerland,1778835602,1778835603,9.0805,49.8557,10675.62,False,224.79,196.49,-2.60,None,10416.54,0745,False,0
588,1778835603,399832,FHGBS,France,1778835596,1778835596,8.5080,49.4729,297.18,False,28.88,274.09,-0.33,None,NaN,None,False,0
589,1778835603,398562,AFR44SA,France,1778835603,1778835603,8.4527,50.1151,3322.32,False,148.30,69.49,-5.53,None,3230.88,1000,False,0


In [14]:
cleaned_data

,time,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,squawk
0,1778835603,3ffc29,DMFSS,Germany,1778835600,1778835602,12.5225,48.4242,1059.18,False,45.79,270.64,-0.33,NaN,7000
1,1778835603,511187,MBU1RD,Estonia,1778835602,1778835602,12.2108,51.4197,NaN,True,5.14,104.06,NaN,NaN,1000
2,1778835603,511171,MBU4HW,Estonia,1778835603,1778835603,10.3395,50.4433,8808.72,False,237.14,34.03,-9.10,8549.64,1000
3,1778835603,511170,MBU1TF,Estonia,1778835603,1778835603,9.4142,49.8205,10980.42,False,245.48,10.63,0.00,10751.82,1000
4,1778835603,3d381b,DETMP,Germany,1778835602,1778835602,12.8749,48.8068,891.54,False,42.79,108.22,0.33,853.44,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586,1778835603,4b17e3,SWR7AK,Switzerland,1778835595,1778835603,8.5678,47.4550,525.78,True,0.00,129.38,NaN,NaN,2000
587,1778835603,4b17fc,SWR2WX,Switzerland,1778835602,1778835603,9.0805,49.8557,10675.62,False,224.79,196.49,-2.60,10416.54,0745
588,1778835603,399832,FHGBS,France,1778835596,1778835596,8.5080,49.4729,297.18,False,28.88,274.09,-0.33,NaN,None
589,1778835603,398562,AFR44SA,France,1778835603,1778835603,8.4527,50.1151,3322.32,False,148.30,69.49,-5.53,3230.88,1000
